<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DeepLearning/blob/main/Interpretabilitymaps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Saliency Maps & Interpretierbarkeit mit Captum

**Lernziele:**
- Verstehen, was Saliency Maps zeigen und was nicht
- Verschiedene Attributionsmethoden kennenlernen (Saliency, Integrated Gradients, GradCAM, SmoothGrad)
- Kritisch reflektieren: Sind diese Erklärungen "echt"? (Sanity Check)

---

## 1. Setup & Installation

In [ ]:
# Captum installieren (Interpretability-Library von Meta für PyTorch)
!pip install captum -q

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import requests
from io import BytesIO
import numpy as np
import matplotlib.pyplot as plt

# Captum Imports
from captum.attr import Saliency, IntegratedGradients, LayerGradCam, NoiseTunnel, LayerAttribution

# Für reproduzierbare Ergebnisse
torch.manual_seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Verwendetes Device: {device}")

## 2. Modell laden

Wir verwenden **ResNet-18** – ein relativ kleines CNN. Es wurde auf ImageNet (1000 Klassen, ca. 1.2 Mio. Bilder) trainiert.

In [ ]:
# Vortrainiertes ResNet-18 laden
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model = model.to(device)
model.eval()  # Wichtig: Evaluation-Modus für Inferenz

print(f"Modell geladen: {sum(p.numel() for p in model.parameters()):,} Parameter")

In [ ]:
# ImageNet Klassennamen laden
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
imagenet_classes = requests.get(LABELS_URL).text.strip().split("\n")
print(f"{len(imagenet_classes)} Klassen geladen. Beispiele: {imagenet_classes[:5]}")

## 3. Hilfsfunktionen

In [ ]:
# Preprocessing für ImageNet-Modelle
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Mittelwert und Standardabweichung als Tensoren (für Baselines im Pixelraum)
mean_t = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
std_t = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)

# Manche Server (z.B. Wikimedia) verlangen einen User-Agent
HEADERS = {"User-Agent": "FHNW-BAI-DeepLearning-Notebook/1.0 (Lehrzwecke)"}

def load_image(url_or_path):
    '''Lädt ein Bild von URL oder lokalem Pfad.'''
    if url_or_path.startswith('http'):
        response = requests.get(url_or_path, headers=HEADERS, timeout=15)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert('RGB')
    else:
        img = Image.open(url_or_path).convert('RGB')
    return img

def predict(model, input_tensor, k=5):
    '''Gibt die Top-k Vorhersagen (Name, Wahrscheinlichkeit, Index) zurück.'''
    with torch.no_grad():
        output = model(input_tensor)
        probs = F.softmax(output, dim=1)
        topk_probs, topk_indices = probs.topk(k)

    results = []
    for prob, idx in zip(topk_probs[0], topk_indices[0]):
        results.append((imagenet_classes[idx], prob.item(), idx.item()))
    return results, topk_indices[0][0].item()

def tensor_to_image(tensor):
    '''Konvertiert normalisierten Tensor zurück zu anzeigbarem Bild.'''
    img = tensor.detach() * std_t + mean_t   # Denormalisieren
    img = img.squeeze().permute(1, 2, 0).cpu().numpy()
    return np.clip(img, 0, 1)

def pixel_baseline(wert):
    '''Einfarbiges Bild im Pixelraum (0 = schwarz, 1 = weiss), danach normalisiert.

    Achtung: Das Modell sieht normalisierte Bilder. Ein Tensor voller Nullen ist
    deshalb NICHT schwarz, sondern das (graue) ImageNet-Mittelwertbild!
    '''
    return ((torch.full((1, 3, 224, 224), float(wert), device=device) - mean_t) / std_t)

def gradcam_map(gradcam_obj, x, target):
    '''GradCAM berechnen (inkl. ReLU) und auf Inputgrösse hochskalieren.'''
    attr = gradcam_obj.attribute(x, target=target, relu_attributions=True)
    attr_up = LayerAttribution.interpolate(attr, (224, 224), interpolate_mode='bilinear')
    return attr_up.squeeze().cpu().detach().numpy()

def attr_to_map(attr):
    '''Pixel-Attribution (1, 3, H, W) -> 2D-Map: Betrag, Maximum über Farbkanäle.'''
    return np.abs(attr.squeeze().cpu().detach().numpy()).max(axis=0)

## 4. Testbild laden

Wir starten mit einem klassischen Beispiel. Später kannst du eigene Bilder testen!

In [ ]:
# Beispielbilder zum Testen (einfach Auswahl ändern oder eigenes Bild hochladen)
EXAMPLE_IMAGES = {
    "hund": "https://images.unsplash.com/photo-1587300003388-59208cc962cb?w=800",
    "katze": "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=800",
    "auto": "https://images.unsplash.com/photo-1494976388531-d1058494ceb8?w=800",
    "vogel": "https://images.unsplash.com/photo-1444464666168-49d633b86797?w=800",
}

# Ersatz-URLs (Wikimedia), falls Unsplash nicht erreichbar ist
FALLBACK_IMAGES = {
    "hund": "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/1200px-YellowLabradorLooking_new.jpg",
    "katze": "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg",
    "auto": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1b/2019_Toyota_Corolla_Icon_Tech_VVT-i_Hybrid_1.8.jpg/1200px-2019_Toyota_Corolla_Icon_Tech_VVT-i_Hybrid_1.8.jpg",
    "vogel": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/45/Eopsaltria_australis_-_Mogo_Campground.jpg/1200px-Eopsaltria_australis_-_Mogo_Campground.jpg",
}

# Bild auswählen
selected_image = "katze"  # <-- HIER ÄNDERN, um ein anderes Bild zu versuchen

# Bild laden (mit Ersatz-Quelle) und vorbereiten
try:
    original_image = load_image(EXAMPLE_IMAGES[selected_image])
except Exception as e:
    print(f"Hauptquelle nicht erreichbar ({e}) – verwende Ersatz-Bild.")
    original_image = load_image(FALLBACK_IMAGES[selected_image])

input_tensor = preprocess(original_image).unsqueeze(0).to(device)
input_tensor.requires_grad = True  # Wichtig für Gradient-basierte Methoden!

# Vorhersage
predictions, top_class = predict(model, input_tensor)

# Anzeigen
plt.figure(figsize=(8, 6))
plt.imshow(original_image)
plt.title(f"Top-Vorhersage: {predictions[0][0]} ({predictions[0][1]*100:.1f}%)")
plt.axis('off')
plt.show()

print("\nTop-5 Vorhersagen:")
for label, prob, idx in predictions:
    print(f"  {label} (Klasse {idx}): {prob*100:.2f}%")

---

## 5. Attributionsmethoden

Jetzt wird es spannend: Wir fragen das Modell "Warum hast du so entschieden?"

### 5.1 Vanilla Saliency (Gradient)

**Idee:** Berechne den Gradienten des Outputs bezüglich des Inputs. Pixel mit hohem Gradienten sind "wichtig" für die Entscheidung.



In [ ]:
# Saliency berechnen
saliency = Saliency(model)
attribution_saliency = saliency.attribute(input_tensor, target=top_class)

# Visualisierung
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Originalbild
axes[0].imshow(tensor_to_image(input_tensor))
axes[0].set_title("Original")
axes[0].axis('off')

# Saliency Map (Graustufen)
saliency_gray = attr_to_map(attribution_saliency)  # Betrag, Max über Farbkanäle
axes[1].imshow(saliency_gray, cmap='hot')
axes[1].set_title("Saliency Map")
axes[1].axis('off')

# Overlay
axes[2].imshow(tensor_to_image(input_tensor))
axes[2].imshow(saliency_gray, cmap='hot', alpha=0.5)
axes[2].set_title("Overlay")
axes[2].axis('off')

plt.suptitle(f"Vanilla Saliency für Klasse: {imagenet_classes[top_class]}", fontsize=14)
plt.tight_layout()
plt.show()

### 💬 Diskussionsfrage

> Schaut euch die Saliency Map an: Fokussiert das Modell auf die "richtigen" Stellen? Oder gibt es überraschende Bereiche?

---

### 5.2 Integrated Gradients

**Problem mit Vanilla Saliency:** Gradienten können bei gesättigten Funktionen verschwinden (Gradient Saturation). Ein Pixel kann wichtig sein, obwohl der lokale Gradient fast null ist.

**Lösung:** Wir gehen schrittweise von einer Baseline $x'$ (z.B. schwarzes Bild) zum Input $x$, berechnen bei jedem Schritt den Gradienten, **mitteln** diese Gradienten und gewichten das Resultat mit der Differenz $x - x'$.

**Formel:** $\text{IG}(x) = (x - x') \cdot \int_0^1 \frac{\partial f(x' + \alpha(x-x'))}{\partial x} d\alpha$

(Das Integral wird mit `n_steps` Stützstellen angenähert – das entspricht dem Mittelwert der Gradienten.)

Diese Methode erfüllt wichtige **Axiome**:
- **Completeness:** Die Attributionen summieren sich genau zur Differenz der Modellausgabe: $\sum_i \text{IG}_i(x) = f(x) - f(x')$. Die "Wichtigkeit" wird also vollständig auf die Pixel verteilt.
- **Sensitivity:** Unterscheiden sich Input und Baseline in nur einem Feature und führt das zu einer anderen Vorhersage, erhält dieses Feature eine Attribution ≠ 0.
- **Implementation Invariance:** Zwei Netzwerke, die für jeden Input denselben Output liefern (auch wenn sie intern verschieden aufgebaut sind), erhalten dieselben Attributionen.

In [ ]:
# Integrated Gradients berechnen
ig = IntegratedGradients(model)

# Baseline: schwarzes Bild IM PIXELRAUM (siehe pixel_baseline in den Hilfsfunktionen)
baseline_black = pixel_baseline(0.0)

attribution_ig, delta = ig.attribute(
    input_tensor,
    baselines=baseline_black,
    target=top_class,
    n_steps=50,                    # Anzahl der Integrationsschritte
    return_convergence_delta=True  # Wie gut ist die Annäherung des Integrals?
)

# Completeness überprüfen: Summe der Attributionen ≈ f(x) - f(Baseline)
with torch.no_grad():
    f_x = model(input_tensor)[0, top_class].item()
    f_base = model(baseline_black)[0, top_class].item()
print(f"f(x) - f(Baseline)         = {f_x - f_base:.3f}  (Logits)")
print(f"Summe der IG-Attributionen = {attribution_ig.sum().item():.3f}")
print(f"Approximationsfehler       = {delta.item():.3f}  (mehr n_steps -> kleiner)")

# Visualisierung
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(tensor_to_image(input_tensor))
axes[0].set_title("Original")
axes[0].axis('off')

ig_gray = attr_to_map(attribution_ig)
axes[1].imshow(ig_gray, cmap='hot')
axes[1].set_title("Integrated Gradients")
axes[1].axis('off')

axes[2].imshow(tensor_to_image(input_tensor))
axes[2].imshow(ig_gray, cmap='hot', alpha=0.5)
axes[2].set_title("Overlay")
axes[2].axis('off')

plt.suptitle(f"Integrated Gradients für Klasse: {imagenet_classes[top_class]}", fontsize=14)
plt.tight_layout()
plt.show()

### 💡 Experiment: Andere Baseline ausprobieren

Die Wahl der Baseline beeinflusst die Attribution! Teste verschiedene Baselines.

**Kleine Falle:** Das Modell sieht *normalisierte* Bilder. Ein Tensor voller Nullen ist darum nicht schwarz, sondern das graue ImageNet-Mittelwertbild. In der ersten Zeile siehst du, wie die Baselines wirklich aussehen.

In [ ]:
# Verschiedene Baselines vergleichen
baselines = {
    "Schwarz": pixel_baseline(0.0),
    "Grau (Nullen =\nImageNet-Mittelwert)": torch.zeros_like(input_tensor),
    "Weiss": pixel_baseline(1.0),
    "Rauschen": ((torch.rand(1, 3, 224, 224, device=device) - mean_t) / std_t),
    "Blur": transforms.GaussianBlur(kernel_size=31, sigma=10)(input_tensor.detach()),
}

fig, axes = plt.subplots(2, len(baselines), figsize=(20, 8))

for col, (name, baseline) in enumerate(baselines.items()):
    attr = ig.attribute(input_tensor, baselines=baseline, target=top_class, n_steps=30)

    # Zeile 1: So sieht die Baseline im Pixelraum aus
    axes[0, col].imshow(tensor_to_image(baseline))
    axes[0, col].set_title(f"Baseline: {name}")
    axes[0, col].axis('off')

    # Zeile 2: Resultierende Attribution
    axes[1, col].imshow(tensor_to_image(input_tensor))
    axes[1, col].imshow(attr_to_map(attr), cmap='hot', alpha=0.5)
    axes[1, col].set_title("IG-Attribution")
    axes[1, col].axis('off')

plt.suptitle("Einfluss der Baseline auf Integrated Gradients", fontsize=14)
plt.tight_layout()
plt.show()

### 💬 Diskussionsfrage

> Welche Baseline ist die "richtige"? Gibt es überhaupt eine richtige Antwort?

---

### 5.3 GradCAM (Gradient-weighted Class Activation Mapping)

**Idee:** Statt Pixel-level Attributionen schauen wir uns an, welche **Feature Maps** in einer bestimmten Schicht wichtig sind.

**Vorteil:** Gröbere, aber oft intuitivere Erklärungen. Zeigt "Regionen" statt einzelner Pixel.

**Welche Schicht?** Wir nehmen den Output des letzten Residual-Blocks (`model.layer4[-1]`). Würden wir z.B. `layer4[-1].conv2` nehmen, lägen wir *vor* BatchNorm, Skip-Connection und ReLU – das ergibt oft unsauberere Maps.

In [ ]:
# GradCAM auf dem Output des letzten Residual-Blocks
gradcam = LayerGradCam(model, model.layer4[-1])

# GradCAM ist bei ResNet-18 nur 7x7 gross -> wird auf 224x224 hochskaliert
# (relu_attributions=True wendet die ReLU aus dem GradCAM-Paper an)
gradcam_np = gradcam_map(gradcam, input_tensor, top_class)

# Visualisierung
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(tensor_to_image(input_tensor))
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(gradcam_np, cmap='jet')
axes[1].set_title("GradCAM Heatmap")
axes[1].axis('off')

axes[2].imshow(tensor_to_image(input_tensor))
axes[2].imshow(gradcam_np, cmap='jet', alpha=0.5)
axes[2].set_title("Overlay")
axes[2].axis('off')

plt.suptitle(f"GradCAM für Klasse: {imagenet_classes[top_class]}", fontsize=14)
plt.tight_layout()
plt.show()

---

## 6. Methodenvergleich

Alle drei Methoden auf einen Blick:

In [ ]:
def compare_methods(input_tensor, target_class):
    '''Vergleicht alle Attributionsmethoden nebeneinander.'''

    # Berechnungen
    sal_np = attr_to_map(saliency.attribute(input_tensor, target=target_class))
    ig_np = attr_to_map(ig.attribute(input_tensor, baselines=pixel_baseline(0.0),
                                     target=target_class, n_steps=50))
    gc_np = gradcam_map(gradcam, input_tensor, target_class)

    # Plot
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))

    axes[0].imshow(tensor_to_image(input_tensor))
    axes[0].set_title(f"Original\n{imagenet_classes[target_class]}")
    axes[0].axis('off')

    axes[1].imshow(tensor_to_image(input_tensor))
    axes[1].imshow(sal_np, cmap='hot', alpha=0.6)
    axes[1].set_title("Saliency\n(Gradient)")
    axes[1].axis('off')

    axes[2].imshow(tensor_to_image(input_tensor))
    axes[2].imshow(ig_np, cmap='hot', alpha=0.6)
    axes[2].set_title("Integrated Gradients\n(Pfad-Integration, schwarze Baseline)")
    axes[2].axis('off')

    axes[3].imshow(tensor_to_image(input_tensor))
    axes[3].imshow(gc_np, cmap='jet', alpha=0.5)
    axes[3].set_title("GradCAM\n(Feature-Map-basiert)")
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

compare_methods(input_tensor, top_class)

---

## 7. 🧪 Experimente zum Selbst-Ausprobieren

### Experiment A: Andere Klasse als Ziel

Was, wenn wir fragen: "Wo im Bild sieht das Modell Hinweise auf eine ANDERE Klasse?"

In [ ]:
# Klassen suchen: gibt (Index, Name) für passende Klassen zurück
def find_class(name):
    matches = [(i, c) for i, c in enumerate(imagenet_classes) if name.lower() in c.lower()]
    return matches[:10]

print("Beispiel-Klassen mit 'cat':")
print(find_class("cat"))

print("\nBeispiel-Klassen mit 'retriever':")
print(find_class("retriever"))

In [ ]:
# Vergleiche: Attribution für Top-Klasse vs. andere Klasse
# Standard: zweitbeste Vorhersage. Oder selbst wählen, z.B. alternative_class = 207 (golden retriever)
alternative_class = None  # <-- HIER EINE KLASSE EINTRAGEN

if alternative_class is None:
    alternative_class = predictions[1][2]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, cls in zip(axes, [top_class, alternative_class]):
    ax.imshow(tensor_to_image(input_tensor))
    ax.imshow(gradcam_map(gradcam, input_tensor, cls), cmap='jet', alpha=0.5)
    ax.set_title(f"GradCAM für: {imagenet_classes[cls]} (Klasse {cls})")
    ax.axis('off')

plt.tight_layout()
plt.show()

### Experiment B: Eigenes Bild hochladen

In [ ]:
# In Google Colab: Bild hochladen
try:
    from google.colab import files
    print("Lade ein Bild hoch...")
    uploaded = files.upload()
except ImportError:
    uploaded = None
    print("Nicht in Colab: Setze stattdessen einen lokalen Pfad, z.B. "
          "custom_image = load_image('mein_bild.jpg'), und führe den Rest der Zelle aus.")

if uploaded:
    filename = list(uploaded.keys())[0]
    custom_image = load_image(filename)
    custom_tensor = preprocess(custom_image).unsqueeze(0).to(device)
    custom_tensor.requires_grad = True

    custom_predictions, custom_top_class = predict(model, custom_tensor)
    print(f"\nVorhersage: {custom_predictions[0][0]} ({custom_predictions[0][1]*100:.1f}%)")

    compare_methods(custom_tensor, custom_top_class)

### Experiment C: Andere Schicht für GradCAM

In [ ]:
# GradCAM auf den Outputs der vier Stufen von ResNet-18
layers = {
    "Layer 1 (früh, 56x56)": model.layer1[-1],
    "Layer 2 (28x28)": model.layer2[-1],
    "Layer 3 (14x14)": model.layer3[-1],
    "Layer 4 (spät, 7x7)": model.layer4[-1],
}

fig, axes = plt.subplots(1, len(layers) + 1, figsize=(20, 4))

axes[0].imshow(tensor_to_image(input_tensor))
axes[0].set_title("Original")
axes[0].axis('off')

for ax, (name, layer) in zip(axes[1:], layers.items()):
    gc = LayerGradCam(model, layer)
    ax.imshow(tensor_to_image(input_tensor))
    ax.imshow(gradcam_map(gc, input_tensor, top_class), cmap='jet', alpha=0.5)
    ax.set_title(name)
    ax.axis('off')

plt.suptitle("GradCAM auf verschiedenen Netzwerk-Tiefen", fontsize=14)
plt.tight_layout()
plt.show()

### 💬 Diskussionsfrage

> Frühe Schichten zeigen feinere Details, späte Schichten grössere Regionen. Welche Erklärung ist "besser"?

---

## 8. Kritische Reflexion

### ⚠️ Grenzen von Saliency Maps

1. **Keine Kausalität:** Saliency Maps zeigen, worauf das Modell *lokal reagiert* – nicht, *warum* es so entscheidet, und schon gar nicht, was in der Welt die Ursache ist
2. **Instabilität:** Kleine Änderungen im Bild → grosse Änderungen in der Map
3. **Faithfulness:** Erklärt die Map wirklich das Modell oder nur unsere Erwartungen?
4. **Baseline-Abhängigkeit:** Bei IG beeinflusst die Baseline das Ergebnis massiv

In [ ]:
# Demo: Instabilität von Saliency Maps
# Kleines Gauss-Rauschen hinzufügen (σ = 0.01 im normalisierten Raum)
noise = torch.randn_like(input_tensor) * 0.01
noisy_tensor = (input_tensor.detach() + noise).requires_grad_(True)

# Saliency berechnen
sal_orig = attr_to_map(saliency.attribute(input_tensor, target=top_class))
sal_noisy = attr_to_map(saliency.attribute(noisy_tensor, target=top_class))

# Wie ähnlich sind die beiden Maps? (Korrelation der Pixelwerte, 1 = identisch)
corr = np.corrcoef(sal_orig.ravel(), sal_noisy.ravel())[0, 1]
print(f"Korrelation Original vs. verrauscht: {corr:.2f}")

# Vergleich
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(sal_orig, cmap='hot')
axes[0].set_title("Saliency: Original")
axes[0].axis('off')

axes[1].imshow(sal_noisy, cmap='hot')
axes[1].set_title("Saliency: + Rauschen (σ = 0.01)")
axes[1].axis('off')

axes[2].imshow(np.abs(sal_orig - sal_noisy), cmap='hot')
axes[2].set_title("Differenz")
axes[2].axis('off')

plt.suptitle("⚠️ Instabilität: Minimales Rauschen → Andere Erklärung", fontsize=14)
plt.tight_layout()
plt.show()

### 🩹 Gegenmittel: SmoothGrad

**Idee:** Statt einer einzigen Saliency Map berechnen wir viele – jeweils mit etwas Rauschen auf dem Input – und **mitteln** sie. Zufällige Ausschläge heben sich auf, was übrig bleibt, ist stabiler.

In Captum lässt sich jede Attributionsmethode mit einem `NoiseTunnel` umhüllen.

In [ ]:
smoothgrad = NoiseTunnel(saliency)

sg_orig = attr_to_map(smoothgrad.attribute(input_tensor, target=top_class,
                                           nt_type='smoothgrad', nt_samples=25, stdevs=0.15))
sg_noisy = attr_to_map(smoothgrad.attribute(noisy_tensor, target=top_class,
                                            nt_type='smoothgrad', nt_samples=25, stdevs=0.15))

corr_sg = np.corrcoef(sg_orig.ravel(), sg_noisy.ravel())[0, 1]
print(f"Korrelation Vanilla Saliency (Original vs. verrauscht): {corr:.2f}")
print(f"Korrelation SmoothGrad       (Original vs. verrauscht): {corr_sg:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(sal_orig, cmap='hot')
axes[0].set_title("Vanilla Saliency")
axes[0].axis('off')

axes[1].imshow(sg_orig, cmap='hot')
axes[1].set_title("SmoothGrad: Original")
axes[1].axis('off')

axes[2].imshow(sg_noisy, cmap='hot')
axes[2].set_title("SmoothGrad: + Rauschen (σ = 0.01)")
axes[2].axis('off')

plt.suptitle("SmoothGrad: Mittelung über verrauschte Kopien", fontsize=14)
plt.tight_layout()
plt.show()

### 🧪 Sanity Check: Erklärt die Map überhaupt das Modell?

Adebayo et al. (2018) haben einen einfachen Test vorgeschlagen: Wir nehmen **dasselbe Netzwerk mit zufälligen Gewichten** – ein Modell, das nichts gelernt hat – und berechnen die gleichen Erklärungen.

- Sieht die Map **fast gleich** aus wie beim trainierten Modell, dann zeigt sie vor allem Strukturen des *Bildes* (z.B. Kanten) und nicht, was das *Modell* gelernt hat.
- Eine vertrauenswürdige Methode sollte sich **deutlich verändern**, wenn das Modell zufällig ist.

In [ ]:
# Gleiche Architektur, aber zufällige Gewichte (nicht trainiert)
torch.manual_seed(0)
random_model = models.resnet18(weights=None).to(device).eval()

sal_random = Saliency(random_model)
gradcam_random = LayerGradCam(random_model, random_model.layer4[-1])

# Erklärungen für dieselbe Klasse berechnen
maps = {
    "Saliency": (attr_to_map(saliency.attribute(input_tensor, target=top_class)),
                 attr_to_map(sal_random.attribute(input_tensor, target=top_class))),
    "GradCAM": (gradcam_map(gradcam, input_tensor, top_class),
                gradcam_map(gradcam_random, input_tensor, top_class)),
}

fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for row, (name, (trained_map, random_map)) in enumerate(maps.items()):
    cmap = 'jet' if name == "GradCAM" else 'hot'
    corr_r = np.corrcoef(trained_map.ravel(), random_map.ravel())[0, 1]

    axes[row, 0].imshow(trained_map, cmap=cmap)
    axes[row, 0].set_title(f"{name}: trainiertes Modell")
    axes[row, 0].axis('off')

    axes[row, 1].imshow(random_map, cmap=cmap)
    axes[row, 1].set_title(f"{name}: zufälliges Modell\n(Korrelation: {corr_r:.2f})")
    axes[row, 1].axis('off')

plt.suptitle("Sanity Check: Trainiertes vs. zufälliges Modell", fontsize=14)
plt.tight_layout()
plt.show()

### 💬 Diskussionsfrage

> Bestehen Saliency und GradCAM hier den Sanity Check? Was würde es für die Praxis bedeuten, wenn eine Erklärung bei einem zufälligen Modell fast gleich aussähe?

**Zum Weiterprobieren:** Im Paper von Adebayo et al. fiel u.a. *Guided Backpropagation* durch – die Maps sahen auch beim zufälligen Modell wie ein schöner Kantenfilter des Bildes aus. Ersetze oben `Saliency` durch `GuidedBackprop` (`from captum.attr import GuidedBackprop`) und vergleiche selbst.

---

## 9. Zusammenfassung

| Methode | Vorteile | Nachteile |
|---------|----------|------------|
| **Saliency** | Schnell, einfach | "Verrauscht", instabil |
| **SmoothGrad** | Stabiler als Vanilla Saliency | Langsamer (viele Durchläufe), zusätzliche Parameter |
| **Integrated Gradients** | Theoretisch fundiert, Axiome (z.B. Completeness) | Baseline-abhängig, langsamer |
| **GradCAM** | Intuitive Regionen, stabil | Grob, primär für CNNs |

### 🎯 Key Takeaways

1. **Keine Methode ist perfekt** – alle haben Trade-offs
2. **Post-hoc ≠ echtes Verstehen** – wir approximieren nur
3. **Erklärungen selbst überprüfen** – z.B. mit Sanity Checks
4. **Mehrere Methoden kombinieren** für robustere Insights
5. **Domain-Experten einbeziehen** – nur sie können Plausibilität bewerten

---

## 10. Weiterführende Ressourcen

- [Captum Dokumentation](https://captum.ai/)
- [Paper: "Axiomatic Attribution for Deep Networks"](https://arxiv.org/abs/1703.01365) (Integrated Gradients)
- [Paper: "Grad-CAM"](https://arxiv.org/abs/1610.02391)
- [Paper: "SmoothGrad: removing noise by adding noise"](https://arxiv.org/abs/1706.03825)
- [Paper: "Sanity Checks for Saliency Maps"](https://arxiv.org/abs/1810.03292)
- [Paper: "Stop Explaining Black Box ML Models..."](https://arxiv.org/abs/1811.10154) (Kritik an Post-hoc Erklärungen)